# Step 9: Memory — 세션 간 영속 컨텍스트

## 학습 목표

- **4가지 메모리 타입** (user, feedback, project, reference)의 의미와 용도를 이해합니다
- **MEMORY.md 인덱스 + 개별 토픽 파일** (YAML frontmatter) 구조를 배웁니다
- 시스템 프롬프트에 **메모리를 주입하는 방법**을 구현합니다

> **왜 메모리가 필요한가?**
> Step 8에서 컨텍스트 압축으로 긴 세션을 처리할 수 있게 되었지만, 세션이 끝나면 모든 맥락이 사라집니다.
> "이 프로젝트는 TypeScript + React 기반이야", "테스트는 Jest를 써야 해" 같은 정보를
> 매 세션마다 다시 알려줘야 한다면 매우 비효율적입니다.
> Claude Code는 이런 정보를 **파일 시스템에 영속적으로 저장**합니다.

---

## Claude Code 소스 분석

### 9-1. 메모리 디렉토리 구조 — src/memdir/memdir.ts

Claude Code는 **MemDir** (Memory Directory) 패턴을 사용합니다:

```
~/.claude/memory/          (또는 프로젝트 루트/.claude/memory/)
├── MEMORY.md              ← 인덱스 파일 (모든 토픽의 이름/설명/타입 목록)
└── topics/
    ├── project-stack.md   ← 개별 토픽 파일 (YAML frontmatter + 마크다운 본문)
    ├── coding-style.md
    └── api-keys-note.md
```

**MEMORY.md** (인덱스 파일) 예시:
```markdown
# MEMORY

총 3개 항목

- **project-stack** (project): 프로젝트 기술 스택 정보
- **coding-style** (feedback): 사용자 코딩 스타일 선호
- **api-keys-note** (user): API 키 관리 방법
```

**토픽 파일** 예시:
```markdown
---
name: project-stack
description: 프로젝트 기술 스택 정보
type: project
---

이 프로젝트는 TypeScript + React 기반입니다.
- 빌드 도구: Vite
- 테스트: Vitest + React Testing Library
- 스타일: Tailwind CSS
- 상태 관리: Zustand
```

### 9-2. 4가지 메모리 타입 — buildMemoryPrompt()

```typescript
// src/memdir/memdir.ts — 4가지 메모리 타입

enum MemoryType {
  USER = "user",           // 사용자가 직접 저장한 메모리
                           // 예: "내 이름은 김철수야", "API 키는 여기에 있어"

  FEEDBACK = "feedback",   // 사용자 피드백에서 자동 추출
                           // 예: "테스트 파일은 __tests__/ 폴더에 넣어줘"

  PROJECT = "project",     // 프로젝트 구조/설정 관련
                           // 예: 기술 스택, 디렉토리 구조, 빌드 명령어

  REFERENCE = "reference", // 참조 문서/코드
                           // 예: API 문서 요약, 코드 컨벤션 가이드
}
```

**자동 메모리 추출** (src/services/extractMemories/):
Claude Code는 대화 중 사용자의 피드백이나 프로젝트 정보를 자동으로 감지하여
메모리로 저장할 수 있습니다. "앞으로 항상 이렇게 해줘" 같은 패턴을 감지합니다.

### 9-3. 크기 제한 — MAX_ENTRYPOINT_LINES, MAX_ENTRYPOINT_BYTES

```typescript
// src/memdir/memdir.ts — 크기 제한

const MAX_ENTRYPOINT_LINES = 200   // MEMORY.md 최대 줄 수
const MAX_ENTRYPOINT_BYTES = 25000 // MEMORY.md 최대 바이트 수
```

메모리는 시스템 프롬프트에 주입되므로, 너무 크면 컨텍스트를 낭비합니다.
따라서 **인덱스 파일(MEMORY.md)에 크기 제한**을 둡니다.
개별 토픽 파일은 필요할 때만 로드할 수 있습니다.

### 9-4. 시스템 프롬프트에 메모리 주입

```typescript
// 시스템 프롬프트 구성 시 메모리를 포함

const systemPrompt = [
  BASE_SYSTEM_PROMPT,           // 기본 지시
  buildMemoryPrompt(memdir),    // ★ 메모리 주입
  toolDescriptions,              // 도구 설명
  userInstructions,              // 사용자 커스텀 지시
].join("\n\n")
```

`buildMemoryPrompt()`는 모든 메모리 항목을 읽어서
`## 항목명\n내용` 형태의 마크다운으로 변환합니다.

In [ ]:
import sys, os
sys.path.insert(0, ".")

from mini_claude.memory import (
    MemoryType,
    MemoryEntry,
    MemoryManager,
    parse_frontmatter,
    MAX_ENTRYPOINT_LINES,
    MAX_ENTRYPOINT_BYTES,
)

# --- 메모리 타입 확인 ---
print("=== 4가지 메모리 타입 ===\n")
type_descriptions = {
    MemoryType.USER: "사용자가 직접 저장 (예: 이름, 선호사항)",
    MemoryType.FEEDBACK: "사용자 피드백에서 추출 (예: '항상 이렇게 해줘')",
    MemoryType.PROJECT: "프로젝트 구조/설정 (예: 기술 스택, 빌드 명령어)",
    MemoryType.REFERENCE: "참조 문서/코드 (예: API 문서 요약)",
}
for mem_type, desc in type_descriptions.items():
    print(f"  {mem_type.value:12s} — {desc}")

print(f"\n=== 크기 제한 ===")
print(f"  MAX_ENTRYPOINT_LINES = {MAX_ENTRYPOINT_LINES}")
print(f"  MAX_ENTRYPOINT_BYTES = {MAX_ENTRYPOINT_BYTES:,}")

# --- MemoryEntry 구조 ---
print(f"\n=== MemoryEntry 예시 ===\n")
entry = MemoryEntry(
    name="project-stack",
    description="프로젝트 기술 스택 정보",
    type=MemoryType.PROJECT,
    content="TypeScript + React 기반\n- 빌드: Vite\n- 테스트: Vitest",
)

print("Frontmatter:")
print(entry.to_frontmatter())
print("인덱스 라인:")
print(entry.to_index_line())

---

## Python으로 구현하기

### 구현 1: MemoryManager — CRUD + 인덱스 관리

`mini_claude/memory.py`의 `MemoryManager`는 메모리의 저장/로드/삭제와
MEMORY.md 인덱스 파일의 자동 관리를 담당합니다.

In [ ]:
import tempfile
from pathlib import Path

# 임시 디렉토리에서 메모리 시스템 테스트
tmpdir = tempfile.mkdtemp(prefix="mini_claude_memory_")
manager = MemoryManager(tmpdir)

print(f"=== 메모리 저장 데모 ===\n")
print(f"메모리 디렉토리: {tmpdir}\n")

# --- 메모리 저장 ---
memories_to_save = [
    MemoryEntry(
        name="project-stack",
        description="프로젝트 기술 스택",
        type=MemoryType.PROJECT,
        content=(
            "이 프로젝트는 Python 3.11 기반의 에이전트 프레임워크입니다.\n"
            "- LLM: Anthropic Claude API\n"
            "- 테스트: pytest\n"
            "- 패키지 관리: Poetry"
        ),
    ),
    MemoryEntry(
        name="coding-conventions",
        description="코딩 컨벤션 및 사용자 선호",
        type=MemoryType.FEEDBACK,
        content=(
            "- 함수 docstring은 반드시 한국어로 작성\n"
            "- type hints 필수\n"
            "- 변수명은 snake_case\n"
            "- 클래스명은 PascalCase"
        ),
    ),
    MemoryEntry(
        name="personal-preferences",
        description="사용자 개인 설정",
        type=MemoryType.USER,
        content=(
            "- 이름: 김개발\n"
            "- 선호 에디터: VS Code\n"
            "- 커밋 메시지는 한국어로 작성"
        ),
    ),
]

for entry in memories_to_save:
    path = manager.save(entry)
    print(f"  저장됨: {entry.name} → {Path(path).name}")

# --- 파일 시스템 확인 ---
print(f"\n=== 생성된 파일 구조 ===\n")
for p in sorted(Path(tmpdir).rglob("*")):
    rel = p.relative_to(tmpdir)
    prefix = "  " if p.parent != Path(tmpdir) else ""
    print(f"  {prefix}{rel}")

# --- 토픽 파일 내용 확인 ---
print(f"\n=== 토픽 파일 내용 (project-stack.md) ===\n")
topic_file = Path(tmpdir) / "topics" / "project-stack.md"
print(topic_file.read_text(encoding="utf-8"))

# --- MEMORY.md 인덱스 확인 ---
print(f"=== MEMORY.md 인덱스 ===\n")
index_file = Path(tmpdir) / "MEMORY.md"
print(index_file.read_text(encoding="utf-8"))

### 구현 2: 메모리 로드 + 시스템 프롬프트 주입

저장된 메모리를 로드하고, `build_prompt()`로 시스템 프롬프트에 주입할 텍스트를 생성합니다.
이 텍스트가 LLM의 시스템 프롬프트에 포함되어, 에이전트가 이전 세션의 맥락을 알 수 있게 됩니다.

In [ ]:
# 새로운 MemoryManager로 로드 테스트 (이전 세션 시뮬레이션)
print("=== 메모리 로드 (새 세션 시뮬레이션) ===\n")

# 새로운 매니저 인스턴스 — 같은 디렉토리를 가리킴
new_session_manager = MemoryManager(tmpdir)
entries = new_session_manager.load()

print(f"로드된 메모리: {len(entries)}개\n")
for entry in entries:
    print(f"  [{entry.type.value:10s}] {entry.name}: {entry.description}")
    print(f"             내용 미리보기: {entry.content[:60]}...")
    print()

# --- 타입별 필터링 ---
print("=== 타입별 필터링 ===\n")
for mem_type in MemoryType:
    filtered = new_session_manager.list_entries(type_filter=mem_type)
    names = [e.name for e in filtered]
    print(f"  {mem_type.value:10s}: {names if names else '(없음)'}")

# --- build_prompt() — 시스템 프롬프트에 주입할 텍스트 ---
print("\n=== build_prompt() 출력 ===\n")
prompt_text = new_session_manager.build_prompt()
print(prompt_text)

print(f"\n--- 프롬프트 크기 ---")
print(f"  줄 수: {len(prompt_text.splitlines())}")
print(f"  바이트: {len(prompt_text.encode('utf-8')):,}")
print(f"  한도: {MAX_ENTRYPOINT_LINES}줄 / {MAX_ENTRYPOINT_BYTES:,}바이트")

---

## 실습: 시스템 프롬프트에 메모리가 주입되는 전체 흐름

실제 에이전트에서 메모리가 어떻게 사용되는지, 시스템 프롬프트를 조립하는 전체 과정을 시뮬레이션합니다.

In [ ]:
# 시스템 프롬프트 조립 시뮬레이션 — Claude Code의 프롬프트 구성 방식

BASE_SYSTEM_PROMPT = """You are an AI coding assistant. You help users write, debug, and improve code.
Follow the user's instructions carefully. Use tools when needed."""

TOOL_DESCRIPTIONS = """Available tools: FileRead, FileWrite, Bash, Grep, Glob"""

def build_full_system_prompt(memory_manager: MemoryManager) -> str:
    """
    Claude Code의 시스템 프롬프트 조립 과정을 시뮬레이션합니다.

    구성 순서:
    1. 기본 시스템 프롬프트
    2. 메모리 (프로젝트 지식, 사용자 선호 등)
    3. 도구 설명
    """
    sections = [BASE_SYSTEM_PROMPT]

    # 메모리 주입
    memory_prompt = memory_manager.build_prompt()
    if memory_prompt:
        sections.append(memory_prompt)

    sections.append(TOOL_DESCRIPTIONS)

    return "\n\n".join(sections)


# 메모리가 포함된 시스템 프롬프트 생성
full_prompt = build_full_system_prompt(new_session_manager)

print("=== 최종 시스템 프롬프트 (메모리 포함) ===\n")
print(full_prompt)

print(f"\n{'=' * 60}")
print(f"전체 프롬프트 크기: {len(full_prompt.encode('utf-8')):,} 바이트")
print(f"\n이 프롬프트가 매 API 호출 시 system 파라미터로 전달됩니다.")
print(f"LLM은 이 메모리를 읽고 '이 프로젝트는 Python 3.11 기반이구나',")
print(f"'docstring은 한국어로 써야 하는구나' 등을 알 수 있습니다.")

# --- 메모리 삭제 테스트 ---
print(f"\n=== 메모리 삭제 테스트 ===\n")
deleted = new_session_manager.delete("personal-preferences")
print(f"'personal-preferences' 삭제: {deleted}")
print(f"남은 메모리: {[e.name for e in new_session_manager.list_entries()]}")

# MEMORY.md가 자동 업데이트되었는지 확인
print(f"\n업데이트된 MEMORY.md:")
print(Path(tmpdir, "MEMORY.md").read_text(encoding="utf-8"))

# --- 임시 디렉토리 정리 ---
import shutil
shutil.rmtree(tmpdir, ignore_errors=True)
print(f"(임시 디렉토리 정리 완료)")

---

## 세션 관리 — 대화 저장과 복구

Claude Code는 메모리 외에도 **세션 자체**를 저장합니다.

- `src/history.ts` — 세션 히스토리 관리
- `/resume` 커맨드로 이전 세션을 이어서 진행할 수 있습니다

메모리가 "프로젝트 지식"을 영속화한다면, 세션은 "대화 흐름"을 영속화합니다.

```typescript
// src/history.ts — SessionStore
interface SessionData {
  sessionId: string
  createdAt: string
  messages: Message[]
  model: string
  totalInputTokens: number
  totalOutputTokens: number
}
```

In [ ]:
import sys, os
sys.path.insert(0, ".")

import json
import tempfile
from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path

@dataclass
class Session:
    """대화 세션 — src/history.ts의 SessionData에 대응"""
    session_id: str
    created_at: str
    messages: list[dict]
    model: str = "claude-sonnet-4-20250514"
    total_input_tokens: int = 0
    total_output_tokens: int = 0


class SessionStore:
    """
    세션 저장/복구 — src/history.ts에 대응.
    
    각 세션은 JSON 파일로 저장되며, /resume 커맨드로 복구할 수 있습니다.
    Claude Code에서는 ~/.claude/projects/{project}/sessions/ 디렉토리를 사용합니다.
    """
    
    def __init__(self, sessions_dir: str):
        self.sessions_dir = Path(sessions_dir)
        self.sessions_dir.mkdir(parents=True, exist_ok=True)
    
    def save(self, session: Session) -> str:
        """세션을 JSON 파일로 저장"""
        file_path = self.sessions_dir / f"{session.session_id}.json"
        data = {
            "session_id": session.session_id,
            "created_at": session.created_at,
            "model": session.model,
            "total_input_tokens": session.total_input_tokens,
            "total_output_tokens": session.total_output_tokens,
            "messages": session.messages,
        }
        file_path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
        return str(file_path)
    
    def load(self, session_id: str) -> Session | None:
        """세션 복구"""
        file_path = self.sessions_dir / f"{session_id}.json"
        if not file_path.exists():
            return None
        data = json.loads(file_path.read_text(encoding="utf-8"))
        return Session(**data)
    
    def list_sessions(self, limit: int = 10) -> list[dict]:
        """최근 세션 목록 반환"""
        sessions = []
        for f in sorted(self.sessions_dir.glob("*.json"), reverse=True):
            data = json.loads(f.read_text(encoding="utf-8"))
            sessions.append({
                "id": data["session_id"],
                "created_at": data["created_at"],
                "messages": len(data["messages"]),
                "model": data.get("model", "unknown"),
            })
            if len(sessions) >= limit:
                break
        return sessions

    def delete(self, session_id: str) -> bool:
        """세션 삭제"""
        file_path = self.sessions_dir / f"{session_id}.json"
        if file_path.exists():
            file_path.unlink()
            return True
        return False


# --- 데모: 세션 저장/복구 ---
tmpdir = tempfile.mkdtemp(prefix="mini_claude_sessions_")
store = SessionStore(tmpdir)

# 세션 1 저장
session1 = Session(
    session_id="sess-2026-04-07-001",
    created_at="2026-04-07T10:30:00",
    messages=[
        {"role": "user", "content": "README.md를 읽어줘"},
        {"role": "assistant", "content": "파일을 읽겠습니다.", 
         "tool_calls": [{"id": "tc1", "name": "Read", "input": {"file_path": "README.md"}}]},
        {"role": "tool_result", "tool_use_id": "tc1", "content": "# Mini Claude Workshop..."},
        {"role": "assistant", "content": "README.md의 내용은 다음과 같습니다..."},
    ],
    total_input_tokens=1500,
    total_output_tokens=800,
)
store.save(session1)

# 세션 2 저장
session2 = Session(
    session_id="sess-2026-04-07-002",
    created_at="2026-04-07T14:15:00",
    messages=[
        {"role": "user", "content": "테스트 코드를 작성해줘"},
        {"role": "assistant", "content": "어떤 모듈의 테스트를 작성할까요?"},
        {"role": "user", "content": "memory.py에 대한 pytest 테스트"},
        {"role": "assistant", "content": "memory.py의 테스트를 작성하겠습니다..."},
    ],
    total_input_tokens=3200,
    total_output_tokens=2100,
)
store.save(session2)

print("=== 저장된 세션 목록 ===\n")
for s in store.list_sessions():
    print(f"  {s['id']}  |  {s['created_at'][:19]}  |  메시지 {s['messages']}개  |  {s['model']}")

print(f"\n=== 세션 복구: sess-2026-04-07-001 ===\n")
restored = store.load("sess-2026-04-07-001")
if restored:
    print(f"  세션 ID: {restored.session_id}")
    print(f"  생성 시간: {restored.created_at}")
    print(f"  메시지 수: {len(restored.messages)}")
    print(f"  토큰: input={restored.total_input_tokens:,}, output={restored.total_output_tokens:,}")
    print(f"  첫 메시지: {restored.messages[0]['content']}")
    print(f"  마지막 메시지: {restored.messages[-1]['content'][:60]}...")

import shutil
shutil.rmtree(tmpdir, ignore_errors=True)

### /resume 커맨드 구현

Claude Code에서 `/resume`은 **LocalCommand**로 구현되어 있습니다 (참고: `src/screens/Resume.tsx`).

- 인자 없이 `/resume` → 최근 세션 목록을 표시합니다
- `/resume sess-001` → 해당 세션의 메시지 히스토리를 에이전트에 복원합니다

세션 목록에서 선택하면, 저장된 메시지 배열이 그대로 에이전트의 대화 히스토리로 로드됩니다.

In [ ]:
class ResumeCommand:
    """
    /resume — 이전 세션을 복구하는 커맨드.
    
    Claude Code에서는 src/screens/Resume.tsx에서 구현되며,
    세션 목록을 보여주고 선택한 세션의 대화 히스토리를 복원합니다.
    """
    name = "resume"
    description = "Resume a previous conversation session"
    aliases = ["r"]
    
    def __init__(self, session_store: SessionStore):
        self.session_store = session_store
    
    def execute(self, args: str) -> str:
        """
        인자 없음 → 세션 목록 표시
        세션 ID → 해당 세션 복구
        """
        if not args.strip():
            # 세션 목록 표시
            sessions = self.session_store.list_sessions()
            if not sessions:
                return "이전 세션이 없습니다."
            
            lines = ["최근 세션 목록:\n"]
            for s in sessions:
                lines.append(
                    f"  {s['id']}  |  {s['created_at'][:19]}  |  "
                    f"메시지 {s['messages']}개  |  {s['model']}"
                )
            lines.append(f"\n사용법: /resume <session-id>")
            return "\n".join(lines)
        
        # 세션 복구
        session = self.session_store.load(args.strip())
        if not session:
            return f"세션을 찾을 수 없습니다: {args.strip()}"
        
        return (
            f"세션 복구 완료: {session.session_id}\n"
            f"  메시지: {len(session.messages)}개\n"
            f"  토큰: input={session.total_input_tokens:,}, output={session.total_output_tokens:,}\n"
            f"  모델: {session.model}\n\n"
            f"이전 대화를 이어서 진행합니다."
        )


# --- 데모 ---
tmpdir = tempfile.mkdtemp(prefix="mini_claude_sessions_")
demo_store = SessionStore(tmpdir)

# 샘플 세션 저장
for i in range(3):
    demo_store.save(Session(
        session_id=f"sess-2026-04-0{7+i}-001",
        created_at=f"2026-04-0{7+i}T{10+i}:00:00",
        messages=[{"role": "user", "content": f"세션 {i+1}의 질문"}] * (i + 2),
        total_input_tokens=(i + 1) * 1000,
        total_output_tokens=(i + 1) * 500,
    ))

resume_cmd = ResumeCommand(demo_store)

print("=== /resume (목록 표시) ===\n")
print(resume_cmd.execute(""))

print(f"\n{'=' * 60}\n")
print("=== /resume sess-2026-04-08-001 ===\n")
print(resume_cmd.execute("sess-2026-04-08-001"))

print(f"\n{'=' * 60}\n")
print("=== /resume unknown-session ===\n")
print(resume_cmd.execute("unknown-session"))

shutil.rmtree(tmpdir, ignore_errors=True)

### 메모리와 세션의 관계

| | 메모리 (Memory) | 세션 (Session) |
|---|---|---|
| **저장 단위** | 토픽별 개별 파일 | 대화 전체를 하나의 JSON |
| **수명** | 영구 (명시적 삭제 전까지) | 일시적 (최근 N개만 유지) |
| **용도** | 프로젝트 지식, 사용자 선호 | 대화 이어하기 (/resume) |
| **크기** | 작음 (MEMORY.md 200줄 제한) | 큼 (전체 메시지 히스토리) |
| **시스템 프롬프트** | 매 호출에 주입됨 | 복구 시에만 메시지로 로드 |

메모리는 **"무엇을 기억할지"**, 세션은 **"어디서 이어갈지"**를 결정합니다.

---

## 핵심 설계 원칙 정리

### 1. 파일 시스템 기반 영속성 (File-based Persistence)
데이터베이스 대신 마크다운 파일을 사용합니다. 사용자가 직접 메모리 파일을 편집하거나 git으로 버전 관리할 수 있습니다. 이것은 Claude Code의 철학 -- "사용자가 제어할 수 있어야 한다"를 반영합니다.

### 2. 인덱스 + 개별 파일 구조
MEMORY.md는 목차(인덱스)이고, 실제 내용은 topics/ 디렉토리의 개별 파일에 있습니다. 인덱스만 로드하면 전체 목록을 빠르게 확인할 수 있고, 필요한 토픽만 선택적으로 로드할 수 있습니다.

### 3. YAML Frontmatter로 메타데이터 관리
각 토픽 파일의 상단에 YAML frontmatter로 이름, 설명, 타입을 기록합니다. 마크다운 에디터에서 편집 가능하면서도 프로그래밍적으로 파싱이 쉬운 형식입니다.

### 4. 크기 제한으로 컨텍스트 보호
MAX_ENTRYPOINT_LINES=200, MAX_ENTRYPOINT_BYTES=25,000으로 메모리가 시스템 프롬프트를 과도하게 차지하지 않게 합니다. Step 8에서 배운 컨텍스트 관리와 같은 맥락입니다.

### 5. 시스템 프롬프트 주입 패턴
메모리를 시스템 프롬프트에 포함시키는 것은 가장 단순하고 효과적인 방법입니다. RAG(검색 증강 생성)처럼 복잡한 파이프라인 없이, LLM이 매 호출마다 프로젝트 맥락을 자연스럽게 참조할 수 있습니다.

### 6. 세션 저장으로 대화 연속성
세션을 JSON으로 저장하여 `/resume`으로 복구할 수 있습니다. 메모리가 '지식'을 영속화한다면, 세션은 '대화 흐름'을 영속화합니다. 두 시스템이 결합되어 사용자는 끊김 없는 경험을 할 수 있습니다.

---

## 다음 Step 미리보기

메모리 시스템으로 세션 간 맥락을 유지할 수 있게 되었습니다.

**Step 10: Skill 시스템**에서는:
- **Skill** -- 재사용 가능한 에이전트 행동 단위
- 시스템 프롬프트, 도구 세트, 메모리를 하나의 **Skill로 패키징**하는 방법
- Slash command(`/commit`, `/review-pr`)로 Skill을 호출하는 패턴

을 배웁니다.